# Local RAG PDF Q&A: Colab quickstart

Runs the full local stack (BGE embeddings, FAISS, Qwen2.5-1.5B-Instruct) on a free GPU.

**Runtime > Change runtime type > T4 GPU** first. It also works on CPU, just more slowly.

In [ ]:
!git clone -q https://github.com/roshan0774/local-rag-pdf-qa.git
%cd local-rag-pdf-qa
!pip install -q -e ".[models,dev]"

## 1. Index the Meridian Mutual report (208 pages)
To use your own file instead, upload it in the Files pane and pass its path to `ingest`.

In [ ]:
!rag-pdf-qa ingest data/meridian_mutual_fy2026_full_report.pdf

## 2. Ask questions

In [ ]:
!rag-pdf-qa ask "What was the combined ratio in 2026 and how did it change from 2025?"

In [ ]:
from rag_pdf_qa import Settings, RAGPipeline
from rag_pdf_qa.ingest import load_index
from rag_pdf_qa.llm import HFChatLLM

s = Settings.from_env()
pipe = RAGPipeline(load_index(s), HFChatLLM(s), s)

for q in ["Why was claim CLM-26-11679 denied?", "Who is the Chief Technology Officer?"]:
    a = pipe.ask(q)
    print(q, "->", a.answer, "| pages:", a.pages)

## 3. Evaluate against the golden set

In [ ]:
!rag-pdf-qa eval eval/golden_meridian.jsonl -k 4 --with-llm --out eval/results.json